# Intermediate 02 - Dispatcher & Routing

In this notebook you will:
- Understand the Dispatcher module
- Practice routing logic in a multi-node environment
- Use `%%kamcmd` to query a live server

---

## What is Dispatcher?

```
         ┌── Node 1 (10.0.0.1) ── 📱
📞 ──── 🖥️ LB ──┤
         └── Node 2 (10.0.0.2) ── 📱
```

- The LB (Load Balancer) distributes incoming SIP requests across nodes
- `ds_select_dst()` picks the target node
- Algorithms: round-robin(0), hash(2), first(4), etc.

## 1. Dispatcher Function Call

When you call `ds_select_dst("1", "4")` in real Kamailio:
1. It looks at destination list for set 1
2. Selects a target using algorithm 4 (first available)
3. Sets `$du` (destination URI)

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=disp1
To: <sip:1002@example.com>

In [ ]:
ds_select_dst("1", "4");
xlog("Selected destination via dispatcher");

## 2. Live Server Query (%%kamcmd)

If a local Kamailio is running, you can query its real-time state.

> **Note**: Kamailio must be running. You'll get an error if it's not.

In [ ]:
%%kamcmd dispatcher.list

In [ ]:
%%kamcmd ul.dump

## 3. Full Routing Flow Simulation

Typical INVITE routing flow:

```
INVITE received → Method check → Dialog check →
Dispatcher select → Record-Route → Relay
```

In [ ]:
%%sip INVITE
From: <sip:1001@10.60.91.199>;tag=flow1
To: <sip:1002@10.60.80.218>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# Step 1: Method validation
if (!is_method("INVITE")) {
    send_reply(405, "Method Not Allowed");
    exit;
}
xlog("Step 1: INVITE confirmed");

In [ ]:
# Step 2: Check if in-dialog
if (has_totag()) {
    xlog("In-dialog request — relay directly");
    route(RELAY);
} else {
    xlog("Initial INVITE — need routing");
}

In [ ]:
# Step 3: Lookup callee location
xlog("Looking up $(ru{uri.user})");
lookup("location");

# Step 4: Record-Route (ensure BYE also goes through LB)
record_route();

# Step 5: Relay
xlog("Relaying to $ru");
t_relay();

## 4. REGISTER Processing Flow

REGISTER stores the phone's current location (IP:port) on the server.

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg1
To: <sip:1001@example.com>
Contact: <sip:1001@192.168.1.100:5060;expires=3600>

In [ ]:
if (is_method("REGISTER")) {
    xlog("REGISTER from $(fu{uri.user}) at $ct");
    save("location");
    send_reply(200, "OK");
}

## 5. Kamailio Instance Control (v0.3)

Control local Kamailio instances from notebook cells.

In [ ]:
%%kamailio status

In [ ]:
%%kamailio start

In [ ]:
%%diff

---

### Summary

- `ds_select_dst()` selects a target node via Dispatcher
- `%%kamcmd` queries live Kamailio state
- `%%kamailio status|start|stop` controls instances
- `%%diff` tracks variable changes

Next: **Advanced/01-dialog-and-failover.ipynb** →